#### MySQL Connection

In [1]:
%load_ext sql
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False

import os
from dotenv import load_dotenv
from sqlalchemy import create_engine
from sqlalchemy.engine.url import URL

load_dotenv()

url = URL.create("mysql+pymysql",
                 username="root",
                 password=os.getenv('MYSQL_PW'),
                 host="localhost")
engine = create_engine(url)
%sql engine

# Many to Many Relationship

In [2]:
%%sql

CREATE DATABASE tv_shows;

""


In [3]:
%%sql

USE tv_shows;

CREATE TABLE reviewers
    (
        id INT AUTO_INCREMENT PRIMARY KEY,
        first_name VARCHAR(50) NOT NULL,
        last_name VARCHAR(50) NOT NULL
    )

""


In [4]:
%%sql

CREATE TABLE series
    (
        id INT AUTO_INCREMENT PRIMARY KEY,
        title VARCHAR(100),
        released_year YEAR,
        genre VARCHAR(100)
    )

""


In [6]:
%%sql

CREATE TABLE reviews
    (
        id INT AUTO_INCREMENT PRIMARY KEY,
        rating DECIMAL(2,1),
        series_id INT,
        reviewer_id INT,
        FOREIGN KEY (series_id) REFERENCES series(id),
        FOREIGN KEY (reviewer_id) REFERENCES reviewers(id)
    )

""


In [7]:
%%sql

INSERT INTO series (title, released_year, genre) VALUES
    ('Archer', 2009, 'Animation'),
    ('Arrested Development', 2003, 'Comedy'),
    ("Bob's Burgers", 2011, 'Animation'),
    ('Bojack Horseman', 2014, 'Animation'),
    ("Breaking Bad", 2008, 'Drama'),
    ('Curb Your Enthusiasm', 2000, 'Comedy'),
    ("Fargo", 2014, 'Drama'),
    ('Freaks and Geeks', 1999, 'Comedy'),
    ('General Hospital', 1963, 'Drama'),
    ('Halt and Catch Fire', 2014, 'Drama'),
    ('Malcolm In The Middle', 2000, 'Comedy'),
    ('Pushing Daisies', 2007, 'Comedy'),
    ('Seinfeld', 1989, 'Comedy'),
    ('Stranger Things', 2016, 'Drama');

""


In [8]:
%%sql

INSERT INTO reviewers (first_name, last_name) VALUES
    ('Thomas', 'Stoneman'),
    ('Wyatt', 'Skaggs'),
    ('Kimbra', 'Masters'),
    ('Domingo', 'Cortes'),
    ('Colt', 'Steele'),
    ('Pinkie', 'Petit'),
    ('Marlon', 'Crafford');

""


In [9]:
%%sql

INSERT INTO reviews(series_id, reviewer_id, rating) VALUES
    (1,1,8.0),(1,2,7.5),(1,3,8.5),(1,4,7.7),(1,5,8.9),
    (2,1,8.1),(2,4,6.0),(2,3,8.0),(2,6,8.4),(2,5,9.9),
    (3,1,7.0),(3,6,7.5),(3,4,8.0),(3,3,7.1),(3,5,8.0),
    (4,1,7.5),(4,3,7.8),(4,4,8.3),(4,2,7.6),(4,5,8.5),
    (5,1,9.5),(5,3,9.0),(5,4,9.1),(5,2,9.3),(5,5,9.9),
    (6,2,6.5),(6,3,7.8),(6,4,8.8),(6,2,8.4),(6,5,9.1),
    (7,2,9.1),(7,5,9.7),
    (8,4,8.5),(8,2,7.8),(8,6,8.8),(8,5,9.3),
    (9,2,5.5),(9,3,6.8),(9,4,5.8),(9,6,4.3),(9,5,4.5),
    (10,5,9.9),
    (13,3,8.0),(13,4,7.2),
    (14,2,8.5),(14,3,8.9),(14,4,8.9);

""


In [15]:
%%sql

SELECT *
FROM reviewers
LIMIT 5;

,id,first_name,last_name
0,1,Thomas,Stoneman
1,2,Wyatt,Skaggs
2,3,Kimbra,Masters
3,4,Domingo,Cortes
4,5,Colt,Steele


In [14]:
%%sql

SELECT *
FROM series
LIMIT 5;

,id,title,released_year,genre
0,1,Archer,2009,Animation
1,2,Arrested Development,2003,Comedy
2,3,Bob's Burgers,2011,Animation
3,4,Bojack Horseman,2014,Animation
4,5,Breaking Bad,2008,Drama


In [13]:
%%sql

SELECT *
FROM reviews
LIMIT 5;

,id,rating,series_id,reviewer_id
0,1,8.0,1,1
1,2,7.5,1,2
2,3,8.5,1,3
3,4,7.7,1,4
4,5,8.9,1,5


# TV Series Challenge #01

In [17]:
%%sql

SELECT 
    s.title,
    r.rating
FROM series as s 
JOIN reviews as r 
ON s.id = r.series_id
LIMIT 10;


,title,rating
0,Archer,8.0
1,Archer,7.5
2,Archer,8.5
3,Archer,7.7
4,Archer,8.9
5,Arrested Development,8.1
6,Arrested Development,6.0
7,Arrested Development,8.0
8,Arrested Development,8.4
9,Arrested Development,9.9


# TV Series Challenge #02

In [22]:
%%sql

SELECT
    s.title,
    ROUND(AVG(r.rating), 1) AS avg_rating
FROM series AS s 
JOIN reviews AS r 
ON s.id = r.series_id
GROUP BY s.title
ORDER BY avg_rating

,title,avg_rating
0,General Hospital,5.4
1,Bob's Burgers,7.5
2,Seinfeld,7.6
3,Bojack Horseman,7.9
4,Archer,8.1
5,Arrested Development,8.1
6,Curb Your Enthusiasm,8.1
7,Freaks and Geeks,8.6
8,Stranger Things,8.8
9,Breaking Bad,9.4


# TV Series Challenge #03

In [24]:
%%sql

SELECT 
    rv.first_name,
    rv.last_name, 
    r.rating
FROM reviewers AS rv 
JOIN reviews AS r 
ON rv.id = r.reviewer_id
LIMIT 10;

,first_name,last_name,rating
0,Thomas,Stoneman,8.0
1,Thomas,Stoneman,8.1
2,Thomas,Stoneman,7.0
3,Thomas,Stoneman,7.5
4,Thomas,Stoneman,9.5
5,Wyatt,Skaggs,7.5
6,Wyatt,Skaggs,7.6
7,Wyatt,Skaggs,9.3
8,Wyatt,Skaggs,6.5
9,Wyatt,Skaggs,8.4


# TV Series Challenge #04
#### Identify the series that have no reviews

In [28]:
%%sql

SELECT 
    s.title
FROM series AS s 
LEFT JOIN reviews AS r 
ON s.id = r.series_id
WHERE r.rating IS NULL

,title
0,Malcolm In The Middle
1,Pushing Daisies


# TV Series Challenge #05

In [31]:
%%sql

SELECT 
    s.genre,
    AVG(r.rating) AS avg_rating
FROM series AS s 
JOIN reviews AS r
ON s.id = r.series_id 
GROUP BY s.genre

,genre,avg_rating
0,Animation,7.86000
1,Comedy,8.16250
2,Drama,8.04375


# TV Series Challenge #06

In [41]:
%%sql

SELECT 
    rv.first_name,
    rv.last_name,
    COUNT(r.id) AS COUNT,
    IFNULL(MIN(r.rating), 0) AS MIN,
    IFNULL(MAX(r.rating), 0) AS MAX,
    IFNULL(AVG(r.rating), 0) AS AVG,
    CASE
        WHEN SUM(r.rating) IS NULL THEN 'INACTIVE'
        ELSE 'ACTIVE'
    END AS STATUS
FROM reviewers AS rv
LEFT JOIN reviews AS r 
ON rv.id = r.reviewer_id
GROUP BY rv.first_name, rv.last_name

,first_name,last_name,COUNT,MIN,MAX,AVG,STATUS
0,Thomas,Stoneman,5,7.0,9.5,8.02000,ACTIVE
1,Wyatt,Skaggs,9,5.5,9.3,7.80000,ACTIVE
2,Kimbra,Masters,9,6.8,9.0,7.98889,ACTIVE
3,Domingo,Cortes,10,5.8,9.1,7.83000,ACTIVE
4,Colt,Steele,10,4.5,9.9,8.77000,ACTIVE
5,Pinkie,Petit,4,4.3,8.8,7.25000,ACTIVE
6,Marlon,Crafford,0,0.0,0.0,0.00000,INACTIVE


# TV Series Challenge #07

In [48]:
%%sql

SELECT 
    s.title,
    r.rating,
    CONCAT(rv.first_name, ' ', rv.last_name) AS reviewer
FROM reviews AS r 
JOIN series AS s 
ON r.series_id = s.id
JOIN reviewers AS rv 
ON r.reviewer_id = rv.id
ORDER BY title
LIMIT 20;


,title,rating,reviewer
0,Archer,8.0,Thomas Stoneman
1,Archer,8.9,Colt Steele
2,Archer,7.7,Domingo Cortes
3,Archer,8.5,Kimbra Masters
4,Archer,7.5,Wyatt Skaggs
5,Arrested Development,8.1,Thomas Stoneman
6,Arrested Development,6.0,Domingo Cortes
7,Arrested Development,8.0,Kimbra Masters
8,Arrested Development,8.4,Pinkie Petit
9,Arrested Development,9.9,Colt Steele
